# Clarity Analytics Center — Pipeline Runner

Runs the full pipeline end to end. Run all cells in order.

The first cell clones the repo and moves into it **only if it doesn't exist already**. <br>
This also handles Colab, which starts in `/content` even when the notebook lives inside 
the repository.

The remaining cells execute each pipeline notebook in place, so each one is saved with
its outputs:

1. `01_storage.ipynb` — creates the SQLite database and the Bronze/Silver/Gold schema
2. `02_ingestion.ipynb` — lands the four raw sources into Bronze
3. `03_processing.ipynb` — transforms Bronze → Silver → Gold (star schema)
4. `04_data_governance.ipynb` — creates data catalog, dictionary and business glossary


Result: `storage/clarity_analytics_center.db`.

In [1]:
import os
import sys
import sqlite3
from pathlib import Path

! pip install tabulate
from tabulate import tabulate

# check if script is being started in repo, if not clone repo and run code
if not os.path.exists("notebooks"):
    !git clone https://github.com/clopezfranco27/Group_4_Clarity_Analytics_Center.git
    %cd Group_4_Clarity_Analytics_Center

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\hilla\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=1800 notebooks/01_storage.ipynb

[NbConvertApp] Converting notebook notebooks/01_storage.ipynb to notebook
C:\Users\hilla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[NbConvertApp] Writing 24257 bytes to notebooks\01_storage.ipynb


In [3]:
!jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=1800 notebooks/02_ingestion.ipynb

[NbConvertApp] Converting notebook notebooks/02_ingestion.ipynb to notebook
C:\Users\hilla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[NbConvertApp] Writing 63990 bytes to notebooks\02_ingestion.ipynb


In [4]:
!jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=1800 notebooks/03_processing.ipynb

[NbConvertApp] Converting notebook notebooks/03_processing.ipynb to notebook
C:\Users\hilla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[NbConvertApp] Writing 72786 bytes to notebooks\03_processing.ipynb


In [5]:
!jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=1800 notebooks/04_data_governance.ipynb

[NbConvertApp] Converting notebook notebooks/04_data_governance.ipynb to notebook
C:\Users\hilla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[NbConvertApp] Writing 139649 bytes to notebooks\04_data_governance.ipynb


In [6]:
# display tables and columns in database

db_path = 'storage/clarity_analytics_center.db'

if not Path(db_path).exists():
    print('database file [{db_path}] doesn\'t exist')
    sys.exit(1)

    
conn = sqlite3.connect(db_path)

cursor = conn.cursor()
cursor.execute(f'SELECT name FROM sqlite_master WHERE type="table" '
                f'AND name != "sqlite_sequence"')
tables = cursor.fetchall()

table_data = []
for table_name_tuple in tables:
    table_name = table_name_tuple[0]

    # get columns
    cursor.execute(f"PRAGMA table_info('{table_name}');")
    columns = cursor.fetchall()
    num_columns = len(columns)

    # get rows
    cursor.execute(f"SELECT COUNT(*) FROM '{table_name}';")
    num_rows = cursor.fetchone()[0]

    table_data.append([table_name, num_columns, num_rows])

table_data.sort(key=lambda x: x[0])
print("Tables in the database:")
print(tabulate(table_data,
        headers=["Table Name", "Number of Columns", "Number of Rows"],
        tablefmt="fancy_grid"))

Tables in the database:
╒════════════════════════════════╤═════════════════════╤══════════════════╕
│ Table Name                     │   Number of Columns │   Number of Rows │
╞════════════════════════════════╪═════════════════════╪══════════════════╡
│ bronze_ice_budget              │                   7 │              300 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_ice_enforcement_metrics │                   7 │               20 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_ice_enforcement_pdfs    │                   6 │              125 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_ice_operations          │                   9 │              400 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_treasury_reconciliation │                  14 │              594 │
├────────────────────────────────┼─────────────────────┼────────

In [7]:
conn.close()